<a id="top"></a>

# Lab L0107: context windows and cost

```yaml
title: "Lab L0107: context windows and cost"
keywords: context window, headroom, cost, pricing, input output asymmetry, staircase, lab
estimated duration: 35
```

> **Lesson:** L01. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L01/objectives.md). Solutions: `L0107_lab_solutions.ipynb`. Pure arithmetic — no API key needed.
>
> **After this lab you can:** estimate window headroom; compute call cost; explain the input/output asymmetry and the history staircase.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Headroom](#2-problem-1--headroom)
- [3. Problem 2 — Match the failure mode](#3-problem-2--match-the-failure-mode)
- [4. Problem 3 — Cost of one call](#4-problem-3--cost-of-one-call)
- [5. Problem 4 — The input/output asymmetry](#5-problem-4--the-inputoutput-asymmetry)
- [6. Problem 5 — The conversation-history staircase](#6-problem-5--the-conversation-history-staircase)
- [7. Self-check](#7-self-check)

## 1. Setup

In [1]:
WINDOW = 200_000  # tokens (Claude Sonnet 4.6)
INPUT_USD_PER_MTOK = 3.00  # illustrative only
OUTPUT_USD_PER_MTOK = 15.00  # illustrative only; output is the expensive direction
print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. Problem 1 — Headroom

Implement `headroom`: tokens left after system + history + input + reserved output. Negative means it won't fit.

In [2]:
def headroom(system: int, history: int, current_input: int, reserved_output: int) -> int:
    return WINDOW - (system + history + current_input + reserved_output)


print("tiny call   :", headroom(20, 0, 15, 500))
print("long history:", headroom(20, 180_000, 15, 500))
print("overflow    :", headroom(20, 205_000, 15, 500))  # expect negative

tiny call   : 199465
long history: 19465
overflow    : -5535


[↑ Back to top](#top)

## 3. Problem 2 — Match the failure mode

Name the failure mode for each: **hard rejection**, **silent truncation**, or **quality degradation**.

1. You send 250k tokens to a 200k-window model and the API returns an error.
2. A framework keeps the chat "working" but the model no longer recalls something from 40 turns ago.
3. A call fits, but answers about details buried mid-document get vaguer.

> _TODO: 1 -> ..., 2 -> ..., 3 -> ..._

[↑ Back to top](#top)

## 4. Problem 3 — Cost of one call

Implement `call_cost_usd` with the per-**million**-token rates, then cost a 1,500-in / 300-out call.

In [3]:
def call_cost_usd(input_tokens: int, output_tokens: int) -> float:
    return (
        input_tokens / 1_000_000 * INPUT_USD_PER_MTOK
        + output_tokens / 1_000_000 * OUTPUT_USD_PER_MTOK
    )


print(f"${call_cost_usd(1500, 300):.5f}")

$0.00900


[↑ Back to top](#top)

## 5. Problem 4 — The input/output asymmetry

Cost (a) 2,000-in/50-out vs (b) 50-in/2,000-out. Which is pricier, and what does that imply for prompt design?

In [4]:
long_in_short_out = call_cost_usd(2000, 50)
short_in_long_out = call_cost_usd(50, 2000)
print(f"long input,  short answer: ${long_in_short_out:.5f}")
print(f"short input, long answer:  ${short_in_long_out:.5f}")
# short-input/long-output is pricier: output costs ~5x input here, so paying the model to
# WRITE a lot beats paying it to READ a lot.

long input,  short answer: $0.00675
short input, long answer:  $0.03015


[↑ Back to top](#top)

## 6. Problem 5 — The conversation-history staircase

5 turns, ~200 input + ~60 output each, whole history re-sent. Print cumulative input and per-turn cost, then the total.

In [5]:
PER_TURN_INPUT = 200
PER_TURN_OUTPUT = 60
cumulative = 0
total = 0.0
for t in range(1, 6):
    cumulative += PER_TURN_INPUT
    cost = call_cost_usd(cumulative, PER_TURN_OUTPUT)
    total += cost
    print(f"turn {t}: input re-sent ~{cumulative:4} tokens -> ${cost:.5f}")
print(f"session total: ${total:.5f}")

turn 1: input re-sent ~ 200 tokens -> $0.00150
turn 2: input re-sent ~ 400 tokens -> $0.00210
turn 3: input re-sent ~ 600 tokens -> $0.00270
turn 4: input re-sent ~ 800 tokens -> $0.00330
turn 5: input re-sent ~1000 tokens -> $0.00390
session total: $0.01350


[↑ Back to top](#top)

## 7. Self-check

Compare against `L0107_lab_solutions.ipynb`. Done when your headroom sign flips on overflow and your staircase total matches.

[↑ Back to top](#top)